In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#  Install all dependencies 

import sys, warnings, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'

print('Installing NLP metrics and core dependencies...')
!pip install -q --upgrade pip
!pip install -q \
    transformers==4.44.2 \
    datasets==2.20.0 \
    accelerate==0.33.0 \
    peft==0.12.0 \
    bitsandbytes==0.43.3 \
    sacrebleu==2.4.2 \
    rouge-score==0.1.2 \
    evaluate==0.4.2 \
    bert-score==0.3.13 \
    jiwer==3.0.4 \
    nltk==3.9.1 \
    sentence-transformers==3.0.1 \
    rank-bm25==0.2.2 \
    tqdm pandas numpy scipy scikit-learn matplotlib seaborn

print('Installing FAISS GPU...')
# Tries CUDA 12 GPU faiss, falls back to CUDA 11 if needed
!pip install -q faiss-gpu-cu12 || pip install faiss-gpu

print('All packages installed successfully ✓')

In [ ]:
!pip install -q -U bitsandbytes

In [1]:
import torch
import transformers
import accelerate
import peft
import sacrebleu
import faiss

print("=" * 55)
print(f"✓ PyTorch version   : {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"✓ Transformers      : {transformers.__version__}")
print(f"✓ SacreBLEU         : {sacrebleu.__version__}")
print(f"✓ FAISS GPU Devices : {faiss.get_num_gpus()}")
print("=" * 55)
print("All core libraries are healthy and ready to go! ✓")

✓ PyTorch version   : 2.10.0+cu128 (CUDA: True)
✓ Transformers      : 4.44.2
✓ SacreBLEU         : 2.4.2
✓ FAISS GPU Devices : 2
All core libraries are healthy and ready to go! ✓


In [2]:
#  Airtight Global Noise & Warning Suppression Protocol 
# This block MUST run at the top of every subsequent cell.
import sys, warnings, logging, os

#  Bypass protobuf/wandb version clashes BEFORE any NLP import
sys.modules['wandb'] = None

#  Suppress all Python warnings (deprecations, user warnings, etc.)
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '0'

#  Kill HuggingFace widget progress bars (prevents "Error displaying widget")
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()

#  Silence all relevant loggers to ERROR-only
_NOISY_LOGGERS = [
    'transformers', 'transformers.generation.utils',
    'transformers.tokenization_utils_base',
    'transformers.modeling_utils',
    'sentence_transformers', 'torch',
    'evaluate', 'datasets', 'peft',
    'accelerate', 'bitsandbytes',
    'filelock', 'urllib3', 'requests', 'absl',
]
for _n in _NOISY_LOGGERS:
    logging.getLogger(_n).setLevel(logging.ERROR)

#  Root-level safety net
logging.basicConfig(level=logging.ERROR)
logging.getLogger().setLevel(logging.ERROR)



In [3]:
#  Global Imports, Configuration & Reproducibility
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
for _n in ['transformers','sentence_transformers','torch','evaluate',
           'transformers.generation.utils','datasets','peft','accelerate']:
    logging.getLogger(_n).setLevel(logging.ERROR)

import gc, json, re, math, time, random
from pathlib import Path
from glob import glob
from copy import deepcopy
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import torch

#  Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

#  GPU Configuration
NUM_GPUS = torch.cuda.device_count()
DEVICE   = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'[CONFIG] GPUs available: {NUM_GPUS}')
for _i in range(NUM_GPUS):
    _mem = torch.cuda.get_device_properties(_i).total_memory / 1e9
    print(f'  GPU {_i}: {torch.cuda.get_device_name(_i)} | {_mem:.1f} GB VRAM')

# Output Paths 
WORK_DIR  = Path('/kaggle/working')
MODEL_DIR = WORK_DIR / 'fine_tuned_dialect_model'
FIG_DIR   = WORK_DIR / 'figures'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

#  Canonical Column Names 
COL_SRC     = 'source'
COL_TGT     = 'target'
COL_DIALECT = 'dialect_type'

print(' Global configuration complete.')

[CONFIG] GPUs available: 2
  GPU 0: Tesla T4 | 15.6 GB VRAM
  GPU 1: Tesla T4 | 15.6 GB VRAM
 Global configuration complete.


In [4]:
# Resolve & Load Dataset Files 

sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

from glob import glob
from pathlib import Path
import pandas as pd

WORK_DIR = Path('/kaggle/working')

# ── Explicit Column Mapping for Kothon Dialect CSVs ──────────
COLUMN_MAP = {
    'dialect_text':    'source',        # Column D in Excel
    'standard_bangla': 'target',        # Column B in Excel
    'dialect_name':    'dialect_type',  # Column E in Excel
}

def _resolve_csv(pattern: str, fallback_dialect: str) -> pd.DataFrame:
    matches = glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find '{pattern}' anywhere under /kaggle/input/")
        
    filepath = matches[0]
    print(f'  [FOUND] {filepath}')
    
    df = pd.read_csv(filepath, encoding='utf-8')
    
    # Rename columns to standard names
    df = df.rename(columns=COLUMN_MAP)
    
    # Ensure dialect_type is populated
    if 'dialect_type' not in df.columns or df['dialect_type'].isna().all():
        df['dialect_type'] = fallback_dialect
        
    # Keep only the required columns and clean text
    df = df[['source', 'target', 'dialect_type']].dropna()
    df['source'] = df['source'].astype(str).str.strip()
    df['target'] = df['target'].astype(str).str.strip()
    df['dialect_type'] = df['dialect_type'].astype(str).str.strip().str.lower()
    
    # Filter empty or corrupt rows
    df = df[(df['source'] != '') & (df['source'] != 'nan') &
            (df['target'] != '') & (df['target'] != 'nan')].reset_index(drop=True)
            
    return df

print('Resolving dataset files...')
df_chitt  = _resolve_csv('chittagonian_preprocessed.csv', 'chittagonian')
df_sylhet = _resolve_csv('sylheti_preprocessed.csv', 'sylheti')
df_all    = pd.concat([df_chitt, df_sylhet], ignore_index=True)

print(f'\n[DATA LOADED SUCCESSFULLY ✓]')
print(f'  • Chittagonian: {len(df_chitt):,} pairs')
print(f'  • Sylheti     : {len(df_sylhet):,} pairs')
print(f'  • Total       : {len(df_all):,} pairs\n')

# Display first 5 samples
df_all.head()

Resolving dataset files...
  [FOUND] /kaggle/input/datasets/tayyabamerituran/dialect/chittagonian_preprocessed.csv
  [FOUND] /kaggle/input/datasets/tayyabamerituran/dialect/sylheti_preprocessed.csv

[DATA LOADED SUCCESSFULLY ✓]
  • Chittagonian: 7,871 pairs
  • Sylheti     : 9,169 pairs
  • Total       : 17,040 pairs



,source,target,dialect_type
0,চাটগাঁইয়া ডঅর চাটগাঁ অঞ্চলর আঞ্চলিক ভাষা।,চাটগাঁইয়া বৃহত্তর চট্টগ্রাম অঞ্চলের আঞ্চলিক ভাষা।,chittagonian
1,বাঁআলা ভাষার হোঁয়ারে চাটগাঁইয়া ভাষার বউত শব্দর...,বাংলা ভাষার সঙ্গে চাটগাঁইয়া ভাষার শাব্দিক সম্প...,chittagonian
2,তয় কেউ কেউ এ ভাষারে ওউগগা স্বাধীন ভাষার মজ্জাদ...,তবে কেউ কেউ একে স্বতন্ত্র ভাষার মর্যাদা দিতে উ...,chittagonian
3,চাটগাঁইয়া আদতে মুখর ভাষা।,চাটগাঁইয়া মূলত মৌখিক ভাষা।,chittagonian
4,ইবা নিজর মতো লেকনর কন কুদরত নাই।,এর নিজস্ব লিপি পদ্ধতি নেই।,chittagonian


In [5]:
import sys, warnings, logging, os
from pathlib import Path
from sklearn.model_selection import train_test_split
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
WORK_DIR = Path('/kaggle/working')
WORK_DIR.mkdir(parents=True, exist_ok=True)
#  90% temp / 10% test (Stratified by dialect_type)
_temp, test_df = train_test_split(
    df_all, test_size=0.10,
    stratify=df_all['dialect_type'], random_state=42
)
#  80% train / 10% val (11.11% of 90% = 10% of total)
train_df, val_df = train_test_split(
    _temp, test_size=0.1111,
    stratify=_temp['dialect_type'], random_state=42
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)
# Save splits to disk
train_df.to_csv(WORK_DIR / 'split_train.csv', index=False)
val_df.to_csv(WORK_DIR   / 'split_val.csv',   index=False)
test_df.to_csv(WORK_DIR  / 'split_test.csv',  index=False)
# 4. Build training vocabulary (Essential for Difficulty Estimator in RAG)
train_vocab = set([w for s in train_df['source'] for w in s.split()])
print('[SPLIT] Stratified splits created and saved:')
for _nm, _df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    _vc = _df['dialect_type'].value_counts()
    print(f'  • {_nm:5s}: {len(_df):5,} rows | Chittagonian: {_vc.get("chittagonian", 0):4,} | Sylheti: {_vc.get("sylheti", 0):4,}')
print(f'\n[VOCAB] Unique training vocabulary size: {len(train_vocab):,} words')
# Expose flat lists for inference
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'
_test_sources   = test_df[COL_SRC].tolist()
_test_targets   = test_df[COL_TGT].tolist()
_test_dialects  = test_df[COL_DIALECT].tolist()
_train_sources  = train_df[COL_SRC].tolist()
_train_targets  = train_df[COL_TGT].tolist()
_train_dialects = train_df[COL_DIALECT].tolist()
print('[SPLIT] Complete ✓')

[SPLIT] Stratified splits created and saved:
  • Train: 13,632 rows | Chittagonian: 6,297 | Sylheti: 7,335
  • Val  : 1,704 rows | Chittagonian:  787 | Sylheti:  917
  • Test : 1,704 rows | Chittagonian:  787 | Sylheti:  917

[VOCAB] Unique training vocabulary size: 22,886 words
[SPLIT] Complete ✓


In [ ]:
# Unified Metric Engine 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','evaluate','bert_score','jiwer','rouge_score','nltk']:
    logging.getLogger(_n).setLevel(logging.ERROR)

import sacrebleu
import evaluate as hf_evaluate
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

from nltk.translate.meteor_score import meteor_score as _nltk_meteor
from bert_score import score as bert_score_fn
import jiwer
import pandas as pd
from typing import List, Dict

# Load Hugging Face ROUGE Metric
_hf_rouge = hf_evaluate.load("rouge")

BERTSCORE_MODEL = 'bert-base-multilingual-cased'  # Standard supported model for Bengali BERTScore


def compute_all_metrics(
    hypotheses: List[str],
    references: List[str],
    compute_bertscore: bool = True,
    bertscore_device: str = 'cuda:0',
) -> Dict[str, float]:
    """
    Compute all 10 evaluation metrics robustly for Bengali.
    """
    _pairs = [(str(h).strip(), str(r).strip()) for h, r in zip(hypotheses, references) if str(h).strip() and str(r).strip()]
    if not _pairs:
        return {m: 0.0 for m in ['bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4', 
                                 'rouge', 'rouge1', 'rouge2', 'rougeL',
                                 'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1']}
    hyps, refs = zip(*_pairs)
    hyps, refs = list(hyps), list(refs)
    res = {}

    # SacreBLEU (Overall Standard BLEU)
    try:
        _sc_std = sacrebleu.corpus_bleu(hyps, [refs], smooth_method='exp', tokenize='flores200')
        res['bleu'] = round(_sc_std.score, 4)
    except Exception:
        res['bleu'] = 0.0

    # Individual N-gram BLEU (1, 2, 3, 4)
    for _ord, _key in [(1, 'bleu1'), (2, 'bleu2'), (3, 'bleu3'), (4, 'bleu4')]:
        try:
            _bleu_obj = sacrebleu.BLEU(max_ngram_order=_ord, tokenize='flores200', smooth_method='exp')
            _sc = _bleu_obj.corpus_score(hyps, [refs])
            res[_key] = round(_sc.score, 4)
        except Exception:
            res[_key] = 0.0

    #  ROUGE 
    try:
        rouge_results = _hf_rouge.compute(
            predictions=hyps, 
            references=refs, 
            tokenizer=lambda x: x.split()  # Splitting Bengali text natively by spaces
        )
        r1 = rouge_results['rouge1'] * 100
        r2 = rouge_results['rouge2'] * 100
        rl = rouge_results['rougeL'] * 100
        
        res['rouge1'] = round(r1, 4)
        res['rouge2'] = round(r2, 4)
        res['rougeL'] = round(rl, 4)
        res['rouge']  = round((r1 + r2 + rl) / 3.0, 4)
    except Exception as e:
        print(f"[ROUGE Warning] {e}")
        res['rouge'], res['rouge1'], res['rouge2'], res['rougeL'] = 0.0, 0.0, 0.0, 0.0

    #  METEOR
    _met = []
    for h, r in zip(hyps, refs):
        try: _met.append(_nltk_meteor([r.split()], h.split()))
        except Exception: _met.append(0.0)
    res['meteor'] = round(float((sum(_met) / len(_met)) * 100), 4)

    # chrF++
    try:
        res['chrfpp'] = round(sacrebleu.corpus_chrf(hyps, [refs], word_order=2).score, 4)
    except Exception:
        res['chrfpp'] = 0.0

    # WER & WIL
    try:
        _wer_out = jiwer.process_words(refs, hyps)
        res['wer'] = round(_wer_out.wer, 4)
        res['wil'] = round(_wer_out.wil, 4)
    except Exception:
        res['wer'], res['wil'] = 1.0, 1.0

    # BERTScore
    if compute_bertscore:
        try:
            _, _, _F1 = bert_score_fn(
                hyps, refs, model_type=BERTSCORE_MODEL,
                lang='bn', device=bertscore_device,
                verbose=False, batch_size=32
            )
            res['bertscore_f1'] = round(float(_F1.mean().item() * 100), 4)
        except Exception as e:
            res['bertscore_f1'] = 0.0
    else:
        res['bertscore_f1'] = float('nan')

    return res


def evaluate_by_dialect(
    hypotheses: List[str],
    references: List[str],
    dialect_labels: List[str],
    system_name: str = 'system',
    compute_bertscore: bool = True,
) -> pd.DataFrame:
    """Compute all 10 metrics per dialect + overall."""
    rows = []
    for _d in ['chittagonian', 'sylheti', 'overall']:
        if _d == 'overall':
            _h, _r = hypotheses, references
        else:
            _idx = [i for i, d in enumerate(dialect_labels) if d == _d]
            _h   = [hypotheses[i] for i in _idx]
            _r   = [references[i] for i in _idx]
        _m = compute_all_metrics(_h, _r, compute_bertscore)
        _m.update({'system': system_name, 'dialect': _d, 'n_samples': len(_h)})
        rows.append(_m)
    return pd.DataFrame(rows)


print('Unified Metric Engine ready (BLEU, BLEU-1/2/3/4, ROUGE, ROUGE-1/2/L, METEOR, chrF++, WER, WIL, BERTScore-F1).')

Unified Metric Engine ready (BLEU, BLEU-1/2/3/4, ROUGE, ROUGE-1/2/L, METEOR, chrF++, WER, WIL, BERTScore-F1).


In [7]:
# Baseline Inference Helpers & Memory Utilities 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate','peft','bitsandbytes']:
    logging.getLogger(_n).setLevel(logging.ERROR)

import gc, torch
from typing import List, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
WORK_DIR = __import__('pathlib').Path('/kaggle/working')
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'


def _flush_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


def _safe_seq2seq(
    model, tokenizer, texts: List[str],
    forced_bos_token_id: Optional[int] = None,
    max_new_tokens: int = 128,
    batch_size: int = 8,
    device: torch.device = DEVICE,
) -> List[str]:
    """OOM-safe batch Seq2Seq inference."""
    model.eval()
    out = []
    for _s in range(0, len(texts), batch_size):
        _batch = texts[_s:_s+batch_size]
        try:
            _enc = tokenizer(_batch, return_tensors='pt', padding=True,
                             truncation=True, max_length=256).to(device)
            _kw = dict(max_new_tokens=max_new_tokens, num_beams=4, early_stopping=True)
            if forced_bos_token_id is not None:
                _kw['forced_bos_token_id'] = forced_bos_token_id
            with torch.no_grad():
                _o = model.generate(**_enc, **_kw)
            out.extend(tokenizer.batch_decode(_o, skip_special_tokens=True))
        except torch.cuda.OutOfMemoryError:
            _flush_gpu()
            for _t in _batch:
                try:
                    _e = tokenizer([_t], return_tensors='pt', truncation=True,
                                   max_length=256).to(device)
                    _kw2 = dict(max_new_tokens=max_new_tokens, num_beams=4)
                    if forced_bos_token_id is not None:
                        _kw2['forced_bos_token_id'] = forced_bos_token_id
                    with torch.no_grad():
                        _o2 = model.generate(**_e, **_kw2)
                    out.append(tokenizer.decode(_o2[0], skip_special_tokens=True))
                except Exception:
                    out.append('')
                _flush_gpu()
    return out


# Few-shot examples (fixed across all models)
_FEW_SHOT_EXAMPLES: List[Tuple[str,str]] = [
    (train_df.iloc[0][COL_SRC], train_df.iloc[0][COL_TGT]),
    (train_df.iloc[1][COL_SRC], train_df.iloc[1][COL_TGT]),
    (train_df.iloc[2][COL_SRC], train_df.iloc[2][COL_TGT]),
]


def _few_shot_prefix() -> str:
    return ' | '.join(f'{s} → {t}' for s, t in _FEW_SHOT_EXAMPLES) + ' || '


def _build_few_shot_prompt(source: str, dialect: str) -> str:
    lines = [f'Translate the following {dialect} dialect into Standard Bangla.\n']
    for i, (s, t) in enumerate(_FEW_SHOT_EXAMPLES, 1):
        lines += [f'Example {i}:', f'  Input:  {s}', f'  Output: {t}\n']
    lines += ['Now translate:', f'  Input:  {source}', '  Output:']
    return '\n'.join(lines)


def _build_cot_prompt(source: str, dialect: str) -> str:
    return (
        f'You are an expert Bangla dialect linguist.\n'
        f'Translate this {dialect} text to Standard Bangla.\n'
        f'Think step-by-step: (1) Identify dialectal words, (2) Map to standard forms, '
        f'(3) Reconstruct the full sentence.\n\n'
        f'Input: {source}\n\nStep-by-step reasoning:\n'
    )


FINETUNE_MODEL_ID = 'csebuetnlp/banglat5'  # Default; overridden by best-baseline selection
print('[BASELINE] Helpers ready.')

[BASELINE] Helpers ready.


In [8]:
# Model 1: NLLB-200-Distilled-600M (Zero-Shot & Few-Shot)
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate']: logging.getLogger(_n).setLevel(logging.ERROR)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
_M = 'facebook/nllb-200-distilled-600M'
print(f'[NLLB] Loading {_M}...')

_tok_n = AutoTokenizer.from_pretrained(_M, src_lang='ben_Beng')
_mdl_n = AutoModelForSeq2SeqLM.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)

#  Universal method for both Fast and Slow NLLB tokenizers:
_bos_n = _tok_n.convert_tokens_to_ids('ben_Beng')

# Zero-Shot
print('[NLLB] Generating Zero-Shot predictions...')
_nllb_zs = _safe_seq2seq(_mdl_n, _tok_n, _test_sources, forced_bos_token_id=_bos_n, batch_size=16)

# Few-Shot (context-prepend for encoder-decoder models)
print('[NLLB] Generating Few-Shot predictions...')
_pfx = _few_shot_prefix()
_nllb_fs = _safe_seq2seq(
    _mdl_n, _tok_n, [_pfx + s for s in _test_sources],
    forced_bos_token_id=_bos_n, batch_size=16
)

print('[NLLB] Evaluating metrics across dialects...')
_nllb_m_zs = evaluate_by_dialect(_nllb_zs, _test_targets, _test_dialects, 'NLLB-600M_ZeroShot')
_nllb_m_fs = evaluate_by_dialect(_nllb_fs, _test_targets, _test_dialects, 'NLLB-600M_FewShot')
test_df['nllb_zs'], test_df['nllb_fs'] = _nllb_zs, _nllb_fs

del _mdl_n, _tok_n
_flush_gpu()

print(f'\n[NLLB RESULTS]')
print(f'  • ZS BLEU-4 overall: {_nllb_m_zs[_nllb_m_zs.dialect=="overall"]["bleu4"].values[0]:.2f}')
print(f'  • FS BLEU-4 overall: {_nllb_m_fs[_nllb_m_fs.dialect=="overall"]["bleu4"].values[0]:.2f}')

[NLLB] Loading facebook/nllb-200-distilled-600M...
[NLLB] Generating Zero-Shot predictions...
[NLLB] Generating Few-Shot predictions...
[NLLB] Evaluating metrics across dialects...

[NLLB RESULTS]
  • ZS BLEU-4 overall: 23.94
  • FS BLEU-4 overall: 0.04


In [ ]:
import pandas as pd
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

# Save NLLB metrics immediately
nllb_metrics_df = pd.concat([_nllb_m_zs, _nllb_m_fs], ignore_index=True)
_col_ord = [
    'system', 'dialect', 'n_samples',
    'bleu','bleu1', 'bleu3', 'bleu4',
    'rouge','rouge1', 'rouge2', 'rougeL',
    'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
]
nllb_metrics_df = nllb_metrics_df[_col_ord]

# Save individual model metrics 
nllb_metrics_df.to_csv(WORK_DIR / 'nllb_baseline_metrics.csv', index=False)
# nllb_metrics_df.to_csv(WORK_DIR / 'comprehensive_baseline_metrics.csv', index=False)

# Save current test predictions dataframe
test_df.to_csv(WORK_DIR / 'test_df_checkpoint.csv', index=False)

print('[CHECKPOINT SAVED] NLLB metrics and predictions written to disk.')

In [9]:
#  Model 2: M2M100-418M 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate']: logging.getLogger(_n).setLevel(logging.ERROR)

import torch
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
_M = 'facebook/m2m100_418M'
print(f'[M2M100] Loading {_M}...')
_tok_m = M2M100Tokenizer.from_pretrained(_M, src_lang='bn', tgt_lang='bn')
_mdl_m = M2M100ForConditionalGeneration.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)
_bos_m = _tok_m.get_lang_id('bn')

_m2m_zs = _safe_seq2seq(_mdl_m, _tok_m, _test_sources, forced_bos_token_id=_bos_m, batch_size=16)
_pfx = _few_shot_prefix()
_m2m_fs = _safe_seq2seq(
    _mdl_m, _tok_m, [_pfx + s for s in _test_sources],
    forced_bos_token_id=_bos_m, batch_size=16
)

_m2m_m_zs = evaluate_by_dialect(_m2m_zs, _test_targets, _test_dialects, 'M2M100-418M_ZeroShot')
_m2m_m_fs = evaluate_by_dialect(_m2m_fs, _test_targets, _test_dialects, 'M2M100-418M_FewShot')
test_df['m2m_zs'], test_df['m2m_fs'] = _m2m_zs, _m2m_fs

del _mdl_m, _tok_m; _flush_gpu()
print(f'[M2M100] ZS BLEU-4 overall: {_m2m_m_zs[_m2m_m_zs.dialect=="overall"]["bleu4"].values[0]:.2f}')

[M2M100] Loading facebook/m2m100_418M...
[M2M100] ZS BLEU-4 overall: 11.31


In [ ]:
_m2m_metrics = pd.concat([_m2m_m_zs, _m2m_m_fs], ignore_index=True)
_m2m_metrics.to_csv(WORK_DIR / 'm2m100_metrics.csv', index=False)

#  Save M2M100 text predictions along with ground truths
_m2m_preds = test_df[[COL_SRC, COL_TGT, COL_DIALECT, 'm2m_zs', 'm2m_fs']]
_m2m_preds.to_csv(WORK_DIR / 'm2m100_predictions.csv', index=False)

print('[M2M100] Saved: /kaggle/working/m2m100_metrics.csv')
print('[M2M100] Saved: /kaggle/working/m2m100_predictions.csv')

In [ ]:
#  Model 3: BanglaT5 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate']: logging.getLogger(_n).setLevel(logging.ERROR)

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
_M = 'csebuetnlp/banglat5'
print(f'[BanglaT5] Loading {_M}...')
_tok_bt = AutoTokenizer.from_pretrained(_M)
_mdl_bt = AutoModelForSeq2SeqLM.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)

_PFX = 'translate dialect to standard: '
_bt5_zs = _safe_seq2seq(
    _mdl_bt, _tok_bt, [_PFX + s for s in _test_sources], batch_size=16
)
_pfx_ctx = _few_shot_prefix()
_bt5_fs = _safe_seq2seq(
    _mdl_bt, _tok_bt, [_PFX + _pfx_ctx + s for s in _test_sources], batch_size=16
)

_bt5_m_zs = evaluate_by_dialect(_bt5_zs, _test_targets, _test_dialects, 'BanglaT5_ZeroShot')
_bt5_m_fs = evaluate_by_dialect(_bt5_fs, _test_targets, _test_dialects, 'BanglaT5_FewShot')
test_df['bt5_zs'], test_df['bt5_fs'] = _bt5_zs, _bt5_fs

FINETUNE_MODEL_ID = _M  # BanglaT5 is the primary fine-tuning candidate

del _mdl_bt, _tok_bt; _flush_gpu()
print(f'[BanglaT5] ZS BLEU-4 overall: {_bt5_m_zs[_bt5_m_zs.dialect=="overall"]["bleu4"].values[0]:.2f}')

[BanglaT5] Loading csebuetnlp/banglat5...
[BanglaT5] ZS BLEU-4 overall: 1.91


In [ ]:
# Model 4: Qwen2.5-7B-Instruct (Native FP16 across 2x T4 GPUs)
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['transformers','accelerate']: logging.getLogger(_n).setLevel(logging.ERROR)

import gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

_M = 'Qwen/Qwen2.5-7B-Instruct'
print(f'[Qwen2.5] Sharding {_M} in pure FP16 across 2x T4 GPUs (No bitsandbytes needed)...')

_tok_q = AutoTokenizer.from_pretrained(_M, trust_remote_code=True)
_mdl_q = AutoModelForCausalLM.from_pretrained(
    _M, 
    torch_dtype=torch.float16, 
    device_map='auto',           # Automatically splits ~7GB on GPU 0 & ~7GB on GPU 1
    trust_remote_code=True,
)
_mdl_q.eval()

print(f"  • Model loaded successfully! Memory footprint sharded across dual GPUs.")

def _qwen_infer(prompts: list, max_new_tokens: int = 150) -> list:
    """Qwen chat-template inference."""
    _sys = 'You are a helpful Bangla dialect translation assistant. Translate the dialect to Standard Bangla accurately.'
    results = []
    first_device = next(_mdl_q.parameters()).device
    for _p in prompts:
        try:
            _msg = [{'role': 'system', 'content': _sys}, {'role': 'user', 'content': _p}]
            _txt = _tok_q.apply_chat_template(_msg, tokenize=False, add_generation_prompt=True)
            _inp = _tok_q([_txt], return_tensors='pt').to(first_device)
            with torch.no_grad():
                _out = _mdl_q.generate(
                    **_inp, max_new_tokens=max_new_tokens,
                    do_sample=False, temperature=None, top_p=None,
                    pad_token_id=_tok_q.eos_token_id,
                )
            _new = _out[0][_inp['input_ids'].shape[1]:]
            _dec = _tok_q.decode(_new, skip_special_tokens=True).strip()
            # Clean CoT reasoning tags if present
            for _marker in ['Final translation:', 'Standard Bangla:', 'Output:']:
                if _marker in _dec:
                    _dec = _dec.split(_marker)[-1].strip()
                    break
            results.append(_dec)
        except Exception:
            results.append('')
        finally:
            _flush_gpu()
    return results

# Benchmark on 100 evaluation samples
_QL = min(100, len(_test_sources))
_q_src, _q_tgt, _q_dia = _test_sources[:_QL], _test_targets[:_QL], _test_dialects[:_QL]

print(f'[Qwen2.5] Running Zero-Shot ({_QL} samples)...')
_qwen_zs = _qwen_infer(
    [f'Translate this {d} dialect text to Standard Bangla.\nInput: {s}\nOutput:'
     for s, d in zip(_q_src, _q_dia)]
)

print(f'[Qwen2.5] Running Few-Shot k=3 ({_QL} samples)...')
_qwen_fs = _qwen_infer([_build_few_shot_prompt(s, d) for s, d in zip(_q_src, _q_dia)])

print(f'[Qwen2.5] Running Chain-of-Thought ({_QL} samples)...')
_qwen_cot = _qwen_infer([_build_cot_prompt(s, d) for s, d in zip(_q_src, _q_dia)], max_new_tokens=250)

print('[Qwen2.5] Computing 10-metric evaluation across dialects...')
_qwen_m_zs  = evaluate_by_dialect(_qwen_zs,  _q_tgt, _q_dia, 'Qwen2.5-7B_ZeroShot')
_qwen_m_fs  = evaluate_by_dialect(_qwen_fs,  _q_tgt, _q_dia, 'Qwen2.5-7B_FewShot')
_qwen_m_cot = evaluate_by_dialect(_qwen_cot, _q_tgt, _q_dia, 'Qwen2.5-7B_CoT')

del _mdl_q, _tok_q
_flush_gpu()

print(f'\n[Qwen2.5 RESULTS]')
print(f'  • ZS  BLEU-4 overall: {_qwen_m_zs[_qwen_m_zs.dialect=="overall"]["bleu4"].values[0]:.2f}')
print(f'  • FS  BLEU-4 overall: {_qwen_m_fs[_qwen_m_fs.dialect=="overall"]["bleu4"].values[0]:.2f}')
print(f'  • CoT BLEU-4 overall: {_qwen_m_cot[_qwen_m_cot.dialect=="overall"]["bleu4"].values[0]:.2f}')

[Qwen2.5] Sharding Qwen/Qwen2.5-7B-Instruct in pure FP16 across 2x T4 GPUs (No bitsandbytes needed)...
  • Model loaded successfully! Memory footprint sharded across dual GPUs.
[Qwen2.5] Running Zero-Shot (100 samples)...
[Qwen2.5] Running Few-Shot k=3 (100 samples)...
[Qwen2.5] Running Chain-of-Thought (100 samples)...
[Qwen2.5] Computing 10-metric evaluation across dialects...

[Qwen2.5 RESULTS]
  • ZS  BLEU-4 overall: 24.98
  • FS  BLEU-4 overall: 29.36
  • CoT BLEU-4 overall: 0.94


In [13]:
#  Consolidate Baseline Results 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

import pandas as pd
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

#  Combine All Baseline Metrics
baseline_metrics_df = pd.concat([
    _nllb_m_zs, _nllb_m_fs,
    _m2m_m_zs,  _m2m_m_fs,
    # _it2_m_zs,  _it2_m_fs,
    _bt5_m_zs,  _bt5_m_fs,
    _qwen_m_zs, _qwen_m_fs, _qwen_m_cot,
], ignore_index=True)

# Strict Column Order (All 17 Metrics)
_col_ord = [
    'system', 'dialect', 'n_samples',
    'bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4',
    'rouge', 'rouge1', 'rouge2', 'rougeL',
    'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
]

baseline_metrics_df = baseline_metrics_df[_col_ord]
baseline_metrics_df.to_csv(WORK_DIR / 'comprehensive_baseline_metrics.csv', index=False)
print(f'[BASELINE] Saved: /kaggle/working/comprehensive_baseline_metrics.csv ({len(baseline_metrics_df)} rows)\n')

#  Configure Pandas to Display Every Single Column and Row
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 1000)

#  Print Clean Formatted Tables by Dialect
for d in ['overall', 'chittagonian', 'sylheti']:
    sub = baseline_metrics_df[baseline_metrics_df['dialect'] == d].sort_values('bleu', ascending=False)
    print("\n" + "="*130)
    print(f"                         [BASELINE METRICS: {d.upper()}] (ALL 17 METRICS)")
    print("="*130)
    print(sub.to_string(index=False))

#  Select & Report Best Seq2Seq Model
_ov = baseline_metrics_df[baseline_metrics_df['dialect'] == 'overall']
_seq2seq_ov = _ov[~_ov['system'].str.contains('Qwen')]
_best_row   = _seq2seq_ov.loc[_seq2seq_ov['bleu'].idxmax()]
BEST_BASELINE_SYSTEM = _best_row['system']

print("\n" + "="*130)
print(f"[SELECTION] Best Seq2Seq Baseline for Fine-Tuning: {BEST_BASELINE_SYSTEM}")
print(f"            BLEU: {_best_row['bleu']:.4f} | BLEU-4: {_best_row['bleu4']:.4f} | chrF++: {_best_row['chrfpp']:.4f} | ROUGE-L: {_best_row['rougeL']:.4f}")
print("="*130)


print("\n" + "#"*130)
print("--- [SAFETY NET: RAW CSV TEXT OF comprehensive_baseline_metrics.csv] ---")
print("#"*130)
print(baseline_metrics_df.to_csv(index=False))
print("#"*130)
print("--- [END OF RAW CSV TEXT] ---")

[BASELINE] Saved: /kaggle/working/comprehensive_baseline_metrics.csv (27 rows)


                         [BASELINE METRICS: OVERALL] (ALL 17 METRICS)
              system dialect  n_samples    bleu   bleu1   bleu2   bleu3   bleu4   rouge  rouge1  rouge2  rougeL  meteor  chrfpp     wer    wil  bertscore_f1
  Qwen2.5-7B_FewShot overall        100 29.3557 51.8141 41.8404 34.7981 29.3557 28.4314 35.6491 14.0946 35.5504 28.5404 42.3618  0.6895 0.8597       83.7976
 Qwen2.5-7B_ZeroShot overall        100 24.9787 46.0162 36.2483 29.8785 24.9787 25.1787 31.9102 11.6603 31.9657 24.6442 40.0007  0.8111 0.8886       82.8665
  NLLB-600M_ZeroShot overall       1704 23.9368 46.1551 35.6374 28.8256 23.9368 22.1068 28.6366  9.6271 28.0566 20.7843 36.4486  0.7879 0.9153       81.9286
M2M100-418M_ZeroShot overall       1704 11.3093 29.5476 20.1474 14.8406 11.3093 13.7238 18.6019  4.2520 18.3176 12.0265 23.4168  1.0339 0.9726       76.0247
    BanglaT5_FewShot overall       1704  3.2055  7.0681  5.1943 

In [ ]:
# Recalculate & Consolidate All Baseline Metrics (4 Models)
import pandas as pd
from pathlib import Path

WORK_DIR = Path('/kaggle/working')

print("[RE-EVALUATE] Recalculating metrics using fixed engine...")

#  NLLB-600M
_nllb_m_zs = evaluate_by_dialect(_nllb_zs, _test_targets, _test_dialects, 'NLLB-600M_ZeroShot')
_nllb_m_fs = evaluate_by_dialect(_nllb_fs, _test_targets, _test_dialects, 'NLLB-600M_FewShot')

#  M2M100-418M
_m2m_m_zs  = evaluate_by_dialect(_m2m_zs,  _test_targets, _test_dialects, 'M2M100-418M_ZeroShot')
_m2m_m_fs  = evaluate_by_dialect(_m2m_fs,  _test_targets, _test_dialects, 'M2M100-418M_FewShot')

#  BanglaT5
_bt5_m_zs  = evaluate_by_dialect(_bt5_zs,  _test_targets, _test_dialects, 'BanglaT5_ZeroShot')
_bt5_m_fs  = evaluate_by_dialect(_bt5_fs,  _test_targets, _test_dialects, 'BanglaT5_FewShot')

#  Qwen2.5-7B (Using length slice for Qwen 100-sample benchmark)
_q_len = len(_qwen_zs)
_qwen_m_zs  = evaluate_by_dialect(_qwen_zs,  _test_targets[:_q_len], _test_dialects[:_q_len], 'Qwen2.5-7B_ZeroShot')
_qwen_m_fs  = evaluate_by_dialect(_qwen_fs,  _test_targets[:_q_len], _test_dialects[:_q_len], 'Qwen2.5-7B_FewShot')
_qwen_m_cot = evaluate_by_dialect(_qwen_cot, _test_targets[:_q_len], _test_dialects[:_q_len], 'Qwen2.5-7B_CoT')

print("[RE-EVALUATE] Consolidation complete ✓")

#  Combine all results
baseline_metrics_df = pd.concat([
    _nllb_m_zs, _nllb_m_fs,
    _m2m_m_zs,  _m2m_m_fs,
    _bt5_m_zs,  _bt5_m_fs,
    _qwen_m_zs, _qwen_m_fs, _qwen_m_cot,
], ignore_index=True)

# Standardize column ordering
_col_ord = [
    'system', 'dialect', 'n_samples',
    'bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4',
    'rouge', 'rouge1', 'rouge2', 'rougeL',
    'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
]

baseline_metrics_df = baseline_metrics_df[_col_ord]
baseline_metrics_df.to_csv(WORK_DIR / 'comprehensive_baseline_metrics.csv', index=False)

print(f'\n[SAVED] /kaggle/working/comprehensive_baseline_metrics.csv ({len(baseline_metrics_df)} rows)\n')

#  Display Overall Systems Comparison Table
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

_ov = baseline_metrics_df[baseline_metrics_df['dialect'] == 'overall']
print("="*105)
print("                                [OVERALL BASELINE COMPARISON]")
print("="*105)
print(_ov[['system', 'bleu', 'bleu4', 'rouge', 'rougeL', 'meteor', 'chrfpp', 'bertscore_f1']].sort_values('bleu', ascending=False).to_string(index=False))
print("="*105)

# Select best Seq2Seq baseline for fine-tuning
_seq2seq_ov = _ov[~_ov['system'].str.contains('Qwen')]
_best_row   = _seq2seq_ov.loc[_seq2seq_ov['bleu'].idxmax()]
BEST_BASELINE_SYSTEM = _best_row['system']
print(f'\n[SELECTION] Best Seq2Seq model for Fine-Tuning: {BEST_BASELINE_SYSTEM} (BLEU: {_best_row["bleu"]:.2f} | chrF++: {_best_row["chrfpp"]:.2f})')

[RE-EVALUATE] Recalculating metrics using fixed engine...
[RE-EVALUATE] Consolidation complete ✓

[SAVED] /kaggle/working/comprehensive_baseline_metrics.csv (27 rows)

                                [OVERALL BASELINE COMPARISON]
              system    bleu   bleu4   rouge  rougeL  meteor  chrfpp  bertscore_f1
  Qwen2.5-7B_FewShot 29.3557 29.3557 28.4314 35.5504 28.5404 42.3618       83.7976
 Qwen2.5-7B_ZeroShot 24.9787 24.9787 25.1787 31.9657 24.6442 40.0007       82.8665
  NLLB-600M_ZeroShot 23.9368 23.9368 22.1068 28.0566 20.7843 36.4486       81.9286
M2M100-418M_ZeroShot 11.3093 11.3093 13.7238 18.3176 12.0265 23.4168       76.0247
    BanglaT5_FewShot  3.2055  3.2055  5.2726  6.7476  9.9787 16.3062       62.6264
   BanglaT5_ZeroShot  1.9056  1.9056  3.0938  4.0479  4.3532  8.4006       60.9365
      Qwen2.5-7B_CoT  0.9396  0.9396  1.7020  2.2646  4.4583  7.5331       54.0361
   NLLB-600M_FewShot  0.0371  0.0371  0.3536  0.5315  0.5631  7.5027       61.0462
 M2M100-418M_FewShot  0

RAG

In [ ]:
#  Bilingual Lexical Gloss Construction
# Resolves OOV Paradox  and Echo-Chamber effect .
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

import re, pandas as pd
from collections import Counter, defaultdict
from typing import Dict, List
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'


def tokenize_bangla(text: str) -> List[str]:
    """Lightweight Bangla tokenizer: extract Unicode Bangla word sequences."""
    return re.findall(r'[\u0980-\u09FF]+', str(text))


def build_bilingual_gloss(
    src_texts: List[str],
    tgt_texts: List[str],
    dialect_labels: List[str],
    min_count: int = 2,
    top_k: int = 2000,
) -> Dict[str, Dict[str, str]]:
    """
    IBM Model-1 style bilingual lexical gloss:
    dialect_word → most probable standard_bangla_word.
    """
    gloss = {'chittagonian': {}, 'sylheti': {}}
    for _d in ['chittagonian', 'sylheti']:
        _idx  = [i for i, d in enumerate(dialect_labels) if d == _d]
        _cooc : Dict[str, Counter] = defaultdict(Counter)
        _freq : Counter = Counter()
        for i in _idx:
            _sw = tokenize_bangla(src_texts[i])
            _tw = tokenize_bangla(tgt_texts[i])
            _freq.update(_sw)
            for sw in _sw:
                for tw in _tw:
                    if sw != tw:  # Only dialectal shifts
                        _cooc[sw][tw] += 1
        _entries = []
        for sw, tw_counts in _cooc.items():
            if _freq[sw] < min_count: continue
            best_tw, best_cnt = tw_counts.most_common(1)[0]
            _entries.append((best_cnt / _freq[sw], sw, best_tw))
        _entries.sort(reverse=True)
        for _, sw, tw in _entries[:top_k]:
            gloss[_d][sw] = tw
    return gloss


print('[GLOSS] Building bilingual lexical gloss...')
BILINGUAL_GLOSS = build_bilingual_gloss(
    _train_sources, _train_targets, _train_dialects,
    min_count=2, top_k=2000,
)

_gloss_rows = [
    {'dialect': d, 'dialect_word': dw, 'standard_word': sw}
    for d, mapping in BILINGUAL_GLOSS.items()
    for dw, sw in mapping.items()
]
pd.DataFrame(_gloss_rows).to_csv(WORK_DIR / 'bilingual_gloss.csv', index=False)

print(f'[GLOSS] Chittagonian: {len(BILINGUAL_GLOSS["chittagonian"]):,} | Sylheti: {len(BILINGUAL_GLOSS["sylheti"]):,}')
print('[GLOSS] Sample entries:')
for dw, sw in list(BILINGUAL_GLOSS['chittagonian'].items())[:5]:
    print(f'  chitt: {dw} → std: {sw}')

[GLOSS] Building bilingual lexical gloss...
[GLOSS] Chittagonian: 2,000 | Sylheti: 2,000
[GLOSS] Sample entries:
  chitt: জনসাধারণ → std: টাকায়
  chitt: য় → std: টাকায়
  chitt: ফুত → std: পুত্র
  chitt: অর্থে → std: আলাদা
  chitt: ৫০ → std: টাকায়


In [ ]:
# Full Retrieval Engine: BGE-M3 + BM25Okapi + Reranker + RRF

import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
for _n in ['sentence_transformers','transformers']: logging.getLogger(_n).setLevel(logging.ERROR)

import gc, torch, numpy as np, re, pandas as pd
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional
from pathlib import Path
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

WORK_DIR = Path('/kaggle/working')
COL_SRC, COL_TGT, COL_DIALECT = 'source', 'target', 'dialect_type'

#  Bilingual Gloss Helper 
def tokenize_bangla(text: str) -> List[str]:
    """Lightweight Bangla tokenizer: extract Unicode Bangla word sequences."""
    return re.findall(r'[\u0980-\u09FF]+', str(text))

def build_bilingual_gloss(
    src_texts: List[str],
    tgt_texts: List[str],
    dialect_labels: List[str],
    min_count: int = 2,
    top_k: int = 2000,
) -> Dict[str, Dict[str, str]]:
    """
    IBM Model-1 style bilingual lexical gloss:
    dialect_word → most probable standard_bangla_word.
    """
    gloss = {'chittagonian': {}, 'sylheti': {}}
    for _d in ['chittagonian', 'sylheti']:
        _idx  = [i for i, d in enumerate(dialect_labels) if d == _d]
        _cooc : Dict[str, Counter] = defaultdict(Counter)
        _freq : Counter = Counter()
        for i in _idx:
            _sw = tokenize_bangla(src_texts[i])
            _tw = tokenize_bangla(tgt_texts[i])
            _freq.update(_sw)
            for sw in _sw:
                for tw in _tw:
                    if sw != tw:  # Only dialectal shifts
                        _cooc[sw][tw] += 1
        _entries = []
        for sw, tw_counts in _cooc.items():
            if _freq[sw] < min_count: continue
            best_tw, best_cnt = tw_counts.most_common(1)[0]
            _entries.append((best_cnt / _freq[sw], sw, best_tw))
        _entries.sort(reverse=True)
        for _, sw, tw in _entries[:top_k]:
            gloss[_d][sw] = tw
    return gloss

print('[GLOSS] Building bilingual lexical gloss...')
BILINGUAL_GLOSS = build_bilingual_gloss(
    _train_sources, _train_targets, _train_dialects,
    min_count=2, top_k=2000,
)

_gloss_rows = [
    {'dialect': d, 'dialect_word': dw, 'standard_word': sw}
    for d, mapping in BILINGUAL_GLOSS.items()
    for dw, sw in mapping.items()
]
pd.DataFrame(_gloss_rows).to_csv(WORK_DIR / 'bilingual_gloss.csv', index=False)
print(f'[GLOSS] Chittagonian: {len(BILINGUAL_GLOSS["chittagonian"]):,} | Sylheti: {len(BILINGUAL_GLOSS["sylheti"]):,}')

# Retrieval Engine Definition
DEVICE_STR = 'cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0'

print('[RETRIEVER] Loading BAAI/bge-m3 dense encoder...')
_bge_m3 = SentenceTransformer('BAAI/bge-m3', device=DEVICE_STR)

print('[RETRIEVER] Loading BAAI/bge-reranker-v2-m3...')
_bge_reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE_STR)


class DialectRetrievalEngine:
    def __init__(self, srcs, tgts, dialects, gloss, encoder, reranker, rrf_k=60):
        self.sources, self.targets, self.dialects = srcs, tgts, dialects
        self.gloss, self.encoder = gloss, encoder
        #  Define both attributes so retrieve() can never miss it
        self.reranker = self._reranker = reranker
        self.rrf_k = rrf_k

        print('[RETRIEVER] Building BM25 indices per dialect...')
        self._bm25 = {}
        self._bm25_idx = {}
        for _d in ['chittagonian', 'sylheti']:
            _idx = [i for i, d in enumerate(dialects) if d == _d]
            _toks = [tokenize_bangla(srcs[i]) for i in _idx]
            self._bm25[_d]     = BM25Okapi(_toks)
            self._bm25_idx[_d] = _idx

        print('[RETRIEVER] Encoding corpus with BGE-M3...')
        self._embs = encoder.encode(
            srcs, batch_size=64,
            normalize_embeddings=True,
            show_progress_bar=False,
        )
        print(f'[RETRIEVER] Corpus embeddings shape: {self._embs.shape}')

    def _rare_density(self, text: str, dialect: str) -> float:
        _w = tokenize_bangla(text)
        if not _w: return 0.0
        return sum(1 for w in _w if w in self.gloss.get(dialect, {})) / len(_w)

    def _select_k(self, density: float) -> int:
        """Critique 4: 0ms overhead for easy samples."""
        if density < 0.15: return 0
        if density < 0.35: return 2
        return 4

    def _dense(self, q: str, n: int) -> List[Tuple[int,float]]:
        _qe = self.encoder.encode([q], normalize_embeddings=True)[0]
        _sc = self._embs @ _qe
        _top = np.argsort(_sc)[::-1][:n]
        return [(int(i), float(_sc[i])) for i in _top]

    def _bm25_ret(self, q: str, dialect: str, n: int) -> List[Tuple[int,float]]:
        _t  = tokenize_bangla(q)
        _sc = self._bm25[dialect].get_scores(_t)
        _top = np.argsort(_sc)[::-1][:n]
        _idx_map = self._bm25_idx[dialect]
        return [(int(_idx_map[j]), float(_sc[j])) for j in _top]

    def _rrf(self, lists, k=60) -> List[int]:
        _scores: Dict[int, float] = defaultdict(float)
        for lst in lists:
            for rank, (idx, _) in enumerate(lst):
                _scores[idx] += 1.0 / (k + rank + 1)
        return sorted(_scores, key=_scores.get, reverse=True)

    def retrieve(self, query: str, dialect: str,
                 force_k: Optional[int] = None) -> Tuple[str, int, float]:
        _density = self._rare_density(query, dialect)
        _k       = force_k if force_k is not None else self._select_k(_density)

        if _k == 0:
            return '', 0, _density

        # Candidate retrieval (Dense + BM25)
        _d_res  = self._dense(query, _k * 5)
        _b_res  = self._bm25_ret(query, dialect, _k * 5)
        _fused  = self._rrf([_d_res, _b_res], k=self.rrf_k)[:_k*3]

        # Cross-encoder reranking
        _pairs    = [(query, self.sources[i]) for i in _fused]
        _rr_sc    = self._reranker.predict(_pairs, show_progress_bar=False)
        _reranked = sorted(zip(_fused, _rr_sc), key=lambda x: x[1], reverse=True)
        _top_k    = [i for i, _ in _reranked[:_k]]

        # Compact context format
        _ctx_parts = [
            f'<ctx> [SRC] {self.sources[i]} [TGT] {self.targets[i]} </ctx>'
            for i in _top_k
        ]

        # Gloss injection
        _words      = tokenize_bangla(query)
        _gloss_hits = [(w, self.gloss[dialect][w])
                       for w in _words if w in self.gloss.get(dialect, {})]
        _gloss_str  = ''
        if _gloss_hits:
            _gpairs    = '; '.join(f'{dw}→{sw}' for dw, sw in _gloss_hits)
            _gloss_str = f'<gloss> {_gpairs} </gloss> '

        _full = _gloss_str + ' '.join(_ctx_parts) + f' [QUERY] {query}'
        return _full.strip(), _k, _density


print('[RETRIEVER] Initializing DialectRetrievalEngine...')
RETRIEVAL_ENGINE = DialectRetrievalEngine(
    srcs=_train_sources, tgts=_train_targets, dialects=_train_dialects,
    gloss=BILINGUAL_GLOSS, encoder=_bge_m3, reranker=_bge_reranker, rrf_k=60,
)

_demo_ctx, _demo_k, _demo_rarity = RETRIEVAL_ENGINE.retrieve(
    _test_sources[0], _test_dialects[0]
)
print(f'\n[RETRIEVER] Demo successful ✓')
print(f'  • rare_density : {_demo_rarity:.2f}')
print(f'  • k_selected   : {_demo_k}')
print(f'  • context sample: {_demo_ctx[:150]}...')

[GLOSS] Building bilingual lexical gloss...
[GLOSS] Chittagonian: 2,000 | Sylheti: 2,000
[RETRIEVER] Loading BAAI/bge-m3 dense encoder...
[RETRIEVER] Loading BAAI/bge-reranker-v2-m3...
[RETRIEVER] Initializing DialectRetrievalEngine...
[RETRIEVER] Building BM25 indices per dialect...
[RETRIEVER] Encoding corpus with BGE-M3...
[RETRIEVER] Corpus embeddings shape: (13632, 1024)

[RETRIEVER] Demo successful ✓
  • rare_density : 0.20
  • k_selected   : 2
  • context sample: <gloss> কীতা→কী </gloss> <ctx> [SRC] আফনে আত দেখতে জানইন? [TGT] আপনি হাত দেখতে জানেন? </ctx> <ctx> [SRC] আফনে আইতা নায়? [TGT] অপনি আসবেন না? </ctx> [Q...


In [17]:
# Full RAG Pipeline Across ALL 4 MODELS 
import sys, warnings, logging, os
sys.modules['wandb'] = None
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
from transformers import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()

import gc, torch, numpy as np, pandas as pd
from pathlib import Path
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    M2M100ForConditionalGeneration, M2M100Tokenizer,
    AutoModelForCausalLM
)

WORK_DIR = Path('/kaggle/working')
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# Patch Attribute Mismatch
RETRIEVAL_ENGINE._reranker = getattr(RETRIEVAL_ENGINE, 'reranker', _bge_reranker)

#  Faithfulness Verifier Class 
class FaithfulnessVerifier:
    def __init__(self, encoder, threshold=0.55):
        self.encoder = encoder
        self.threshold = threshold

    def verify_batch(self, sources: list, candidate_preds: list, fallback_preds: list) -> list:
        src_embs  = self.encoder.encode(sources, normalize_embeddings=True, show_progress_bar=False)
        cand_embs = self.encoder.encode(candidate_preds, normalize_embeddings=True, show_progress_bar=False)
        sims = np.sum(src_embs * cand_embs, axis=1)
        
        verified_out = []
        for src, cand, fallback, sim in zip(sources, candidate_preds, fallback_preds, sims):
            if sim >= self.threshold and len(str(cand).strip()) > 0:
                verified_out.append(cand)
            else:
                verified_out.append(fallback)
        return verified_out

verifier = FaithfulnessVerifier(_bge_m3, threshold=0.55)

# Prepare RAG Input Sequences
print("[RAG Engine] Preparing retrieval-augmented inputs for test set...")

bm25_rag_inputs, dense_rag_inputs, hybrid_rag_inputs, adaptive_rag_inputs = [], [], [], []
adaptive_k_list = []

for s, d in zip(_test_sources, _test_dialects):
    # BM25 RAG (k=4)
    bm_res = RETRIEVAL_ENGINE._bm25_ret(s, d, 4)
    ctx_bm = [f"<ctx> [SRC] {RETRIEVAL_ENGINE.sources[i]} [TGT] {RETRIEVAL_ENGINE.targets[i]} </ctx>" for i, _ in bm_res]
    bm25_rag_inputs.append(f"{' '.join(ctx_bm)} [QUERY] {s}".strip())

    # Dense RAG (k=4)
    d_res = RETRIEVAL_ENGINE._dense(s, 4)
    ctx_d = [f"<ctx> [SRC] {RETRIEVAL_ENGINE.sources[i]} [TGT] {RETRIEVAL_ENGINE.targets[i]} </ctx>" for i, _ in d_res]
    dense_rag_inputs.append(f"{' '.join(ctx_d)} [QUERY] {s}".strip())

    # Hybrid RAG + Reranker (k=4)
    ctx_hyb, _, _ = RETRIEVAL_ENGINE.retrieve(s, d, force_k=4)
    hybrid_rag_inputs.append(ctx_hyb)

    # Adaptive RAG (Dynamic k ∈ {0, 2, 4})
    ctx_ad, k_used, _ = RETRIEVAL_ENGINE.retrieve(s, d, force_k=None)
    adaptive_rag_inputs.append(ctx_ad if k_used > 0 else s)
    adaptive_k_list.append(k_used)

print("[RAG Engine] RAG inputs constructed ✓")

all_rag_metrics_rows = []
rag_preds_df = test_df[[COL_SRC, COL_TGT, COL_DIALECT]].copy()



# MODEL 1: NLLB-600M RAG PIPELINE
print("\n" + "="*80)
print("[1/4] Running RAG Pipeline on NLLB-600M...")
print("="*80)
_M = 'facebook/nllb-200-distilled-600M'
_tok_n = AutoTokenizer.from_pretrained(_M, src_lang='ben_Beng')
_mdl_n = AutoModelForSeq2SeqLM.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)
_bos_n = _tok_n.convert_tokens_to_ids('ben_Beng')

nllb_bm25_p   = _safe_seq2seq(_mdl_n, _tok_n, bm25_rag_inputs, forced_bos_token_id=_bos_n, batch_size=16)
nllb_dense_p  = _safe_seq2seq(_mdl_n, _tok_n, dense_rag_inputs, forced_bos_token_id=_bos_n, batch_size=16)
nllb_hybrid_p = _safe_seq2seq(_mdl_n, _tok_n, hybrid_rag_inputs, forced_bos_token_id=_bos_n, batch_size=16)
nllb_adapt_c  = _safe_seq2seq(_mdl_n, _tok_n, adaptive_rag_inputs, forced_bos_token_id=_bos_n, batch_size=16)
nllb_verif_p  = verifier.verify_batch(_test_sources, nllb_adapt_c, _nllb_zs)

rag_preds_df["nllb_bm25"]   = nllb_bm25_p
rag_preds_df["nllb_dense"]  = nllb_dense_p
rag_preds_df["nllb_hybrid"] = nllb_hybrid_p
rag_preds_df["nllb_adapt"]  = nllb_verif_p

all_rag_metrics_rows.extend([
    evaluate_by_dialect(nllb_bm25_p,  _test_targets, _test_dialects, 'NLLB_BM25-RAG'),
    evaluate_by_dialect(nllb_dense_p, _test_targets, _test_dialects, 'NLLB_Dense-RAG'),
    evaluate_by_dialect(nllb_hybrid_p,_test_targets, _test_dialects, 'NLLB_Hybrid-RAG+Reranker'),
    evaluate_by_dialect(nllb_verif_p, _test_targets, _test_dialects, 'NLLB_Adaptive-RAG+Verifier'),
])

del _mdl_n, _tok_n; _flush_gpu()


# MODEL 2: M2M100-418M RAG PIPELINE
print("\n" + "="*80)
print("[2/4] Running RAG Pipeline on M2M100-418M...")
print("="*80)
_M = 'facebook/m2m100_418M'
_tok_m = M2M100Tokenizer.from_pretrained(_M, src_lang='bn', tgt_lang='bn')
_mdl_m = M2M100ForConditionalGeneration.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)
_bos_m = _tok_m.get_lang_id('bn')

m2m_bm25_p   = _safe_seq2seq(_mdl_m, _tok_m, bm25_rag_inputs, forced_bos_token_id=_bos_m, batch_size=16)
m2m_dense_p  = _safe_seq2seq(_mdl_m, _tok_m, dense_rag_inputs, forced_bos_token_id=_bos_m, batch_size=16)
m2m_hybrid_p = _safe_seq2seq(_mdl_m, _tok_m, hybrid_rag_inputs, forced_bos_token_id=_bos_m, batch_size=16)
m2m_adapt_c  = _safe_seq2seq(_mdl_m, _tok_m, adaptive_rag_inputs, forced_bos_token_id=_bos_m, batch_size=16)
m2m_verif_p  = verifier.verify_batch(_test_sources, m2m_adapt_c, _m2m_zs)

rag_preds_df["m2m_bm25"]   = m2m_bm25_p
rag_preds_df["m2m_dense"]  = m2m_dense_p
rag_preds_df["m2m_hybrid"] = m2m_hybrid_p
rag_preds_df["m2m_adapt"]  = m2m_verif_p

all_rag_metrics_rows.extend([
    evaluate_by_dialect(m2m_bm25_p,  _test_targets, _test_dialects, 'M2M100_BM25-RAG'),
    evaluate_by_dialect(m2m_dense_p, _test_targets, _test_dialects, 'M2M100_Dense-RAG'),
    evaluate_by_dialect(m2m_hybrid_p,_test_targets, _test_dialects, 'M2M100_Hybrid-RAG+Reranker'),
    evaluate_by_dialect(m2m_verif_p, _test_targets, _test_dialects, 'M2M100_Adaptive-RAG+Verifier'),
])

del _mdl_m, _tok_m; _flush_gpu()


# MODEL 3: BanglaT5 RAG PIPELINE
print("\n" + "="*80)
print("[3/4] Running RAG Pipeline on BanglaT5...")
print("="*80)
_M = 'csebuetnlp/banglat5'
_tok_bt = AutoTokenizer.from_pretrained(_M)
_mdl_bt = AutoModelForSeq2SeqLM.from_pretrained(_M, torch_dtype=torch.float16).to(DEVICE)
_PFX = 'translate dialect to standard: '

bt5_bm25_p   = _safe_seq2seq(_mdl_bt, _tok_bt, [_PFX + s for s in bm25_rag_inputs], batch_size=16)
bt5_dense_p  = _safe_seq2seq(_mdl_bt, _tok_bt, [_PFX + s for s in dense_rag_inputs], batch_size=16)
bt5_hybrid_p = _safe_seq2seq(_mdl_bt, _tok_bt, [_PFX + s for s in hybrid_rag_inputs], batch_size=16)
bt5_adapt_c  = _safe_seq2seq(_mdl_bt, _tok_bt, [_PFX + s for s in adaptive_rag_inputs], batch_size=16)
bt5_verif_p  = verifier.verify_batch(_test_sources, bt5_adapt_c, _bt5_zs)

rag_preds_df["bt5_bm25"]   = bt5_bm25_p
rag_preds_df["bt5_dense"]  = bt5_dense_p
rag_preds_df["bt5_hybrid"] = bt5_hybrid_p
rag_preds_df["bt5_adapt"]  = bt5_verif_p

all_rag_metrics_rows.extend([
    evaluate_by_dialect(bt5_bm25_p,  _test_targets, _test_dialects, 'BanglaT5_BM25-RAG'),
    evaluate_by_dialect(bt5_dense_p, _test_targets, _test_dialects, 'BanglaT5_Dense-RAG'),
    evaluate_by_dialect(bt5_hybrid_p,_test_targets, _test_dialects, 'BanglaT5_Hybrid-RAG+Reranker'),
    evaluate_by_dialect(bt5_verif_p, _test_targets, _test_dialects, 'BanglaT5_Adaptive-RAG+Verifier'),
])

del _mdl_bt, _tok_bt; _flush_gpu()


# MODEL 4: Qwen2.5-7B RAG PIPELINE (100 Benchmark Samples)
print("\n" + "="*80)
print("[4/4] Running RAG Pipeline on Qwen2.5-7B (Sharded FP16)...")
print("="*80)
_M = 'Qwen/Qwen2.5-7B-Instruct'
_tok_q = AutoTokenizer.from_pretrained(_M, trust_remote_code=True)
_mdl_q = AutoModelForCausalLM.from_pretrained(_M, torch_dtype=torch.float16, device_map='auto', trust_remote_code=True)
_mdl_q.eval()

_QL = min(100, len(_test_sources))
_q_src, _q_tgt, _q_dia = _test_sources[:_QL], _test_targets[:_QL], _test_dialects[:_QL]

qwen_bm25_p   = _qwen_infer([f"Use retrieved context to translate to Standard Bangla:\n{bm25_rag_inputs[i]}" for i in range(_QL)])
qwen_dense_p  = _qwen_infer([f"Use retrieved context to translate to Standard Bangla:\n{dense_rag_inputs[i]}" for i in range(_QL)])
qwen_hybrid_p = _qwen_infer([f"Use retrieved context to translate to Standard Bangla:\n{hybrid_rag_inputs[i]}" for i in range(_QL)])
qwen_adapt_c  = _qwen_infer([f"Translate to Standard Bangla:\n{adaptive_rag_inputs[i]}" for i in range(_QL)])
qwen_verif_p  = verifier.verify_batch(_q_src, qwen_adapt_c, _qwen_zs[:_QL])

all_rag_metrics_rows.extend([
    evaluate_by_dialect(qwen_bm25_p,  _q_tgt, _q_dia, 'Qwen2.5-7B_BM25-RAG'),
    evaluate_by_dialect(qwen_dense_p, _q_tgt, _q_dia, 'Qwen2.5-7B_Dense-RAG'),
    evaluate_by_dialect(qwen_hybrid_p,_q_tgt, _q_dia, 'Qwen2.5-7B_Hybrid-RAG+Reranker'),
    evaluate_by_dialect(qwen_verif_p, _q_tgt, _q_dia, 'Qwen2.5-7B_Adaptive-RAG+Verifier'),
])

del _mdl_q, _tok_q; _flush_gpu()


# CONSOLIDATE AND SAVE ALL RAG RESULTS
rag_preds_df.to_csv(WORK_DIR / "all_models_rag_predictions.csv", index=False)
print(f"\n[SAVED] Predictions saved to: /kaggle/working/all_models_rag_predictions.csv ✓")

all_rag_df = pd.concat([baseline_metrics_df] + all_rag_metrics_rows, ignore_index=True)
_col_ord = [
    'system', 'dialect', 'n_samples',
    'bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4',
    'rouge', 'rouge1', 'rouge2', 'rougeL',
    'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
]
all_rag_df = all_rag_df[_col_ord]
all_rag_df.to_csv(WORK_DIR / "all_models_rag_ablation_metrics.csv", index=False)
print(f"[SAVED] Metrics saved to: /kaggle/working/all_models_rag_ablation_metrics.csv ✓")

# PRINT ALL 17 METRICS FOR OVERALL, CHITTAGONIAN & SYLHETI
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

for d in ['overall', 'chittagonian', 'sylheti']:
    sub_df = all_rag_df[all_rag_df['dialect'] == d].sort_values('bleu', ascending=False)
    print("\n" + "="*125)
    print(f"                                [METRICS TABLE: {d.upper()}] (ALL 17 METRICS)")
    print("="*125)
    print(sub_df.to_string(index=False))

#  PRINT RAW CSV STRING

print("\n" + "#"*125)
print("--- [SAFETY NET: RAW CSV TEXT OF ALL METRICS] ---")
print("#"*125)
print(all_rag_df.to_csv(index=False))
print("#"*125)
print("--- [END OF RAW CSV TEXT] ---")

[RAG Engine] Preparing retrieval-augmented inputs for test set...
[RAG Engine] RAG inputs constructed ✓

[1/4] Running RAG Pipeline on NLLB-600M...

[2/4] Running RAG Pipeline on M2M100-418M...

[3/4] Running RAG Pipeline on BanglaT5...

[4/4] Running RAG Pipeline on Qwen2.5-7B (Sharded FP16)...

[SAVED] Predictions saved to: /kaggle/working/all_models_rag_predictions.csv ✓
[SAVED] Metrics saved to: /kaggle/working/all_models_rag_ablation_metrics.csv ✓

                                [METRICS TABLE: OVERALL] (ALL 17 METRICS)
                          system dialect  n_samples    bleu   bleu1   bleu2   bleu3   bleu4   rouge  rouge1  rouge2  rougeL  meteor  chrfpp     wer    wil  bertscore_f1
  Qwen2.5-7B_Hybrid-RAG+Reranker overall        100 41.8337 57.7591 51.0408 45.9430 41.8337 48.1385 54.8608 34.7903 54.7643 51.0686 56.2218  0.5770 0.7245       86.7627
            Qwen2.5-7B_Dense-RAG overall        100 38.3209 55.1475 47.8309 42.4506 38.3209 41.6951 49.2896 26.9769 48.8189 43.948

In [ ]:
# # CONSOLIDATE AND SAVE ALL RAG RESULTS
# rag_preds_df.to_csv(WORK_DIR / "all_models_rag_predictions.csv", index=False)
# print(f"\n[SAVED] Predictions saved to: /kaggle/working/all_models_rag_predictions.csv ✓")

# all_rag_df = pd.concat([baseline_metrics_df] + all_rag_metrics_rows, ignore_index=True)
# _col_ord = [
#     'system', 'dialect', 'n_samples',
#     'bleu', 'bleu1', 'bleu2', 'bleu3', 'bleu4',
#     'rouge', 'rouge1', 'rouge2', 'rougeL',
#     'meteor', 'chrfpp', 'wer', 'wil', 'bertscore_f1'
# ]
# all_rag_df = all_rag_df[_col_ord]
# all_rag_df.to_csv(WORK_DIR / "all_models_rag_ablation_metrics.csv", index=False)
# print(f"[SAVED] Metrics saved to: /kaggle/working/all_models_rag_ablation_metrics.csv ✓")

# # Print Master Comparative Table
# pd.set_option('display.max_columns', None)
# pd.set_option('display.width', 1000)

# _ov_master = all_rag_df[all_rag_df['dialect'] == 'overall']
# print("\n" + "="*115)
# print("                                [MASTER COMPARATIVE METRICS TABLE: ALL MODELS & RAG VARIANTS]")
# print("="*115)
# print(_ov_master[['system', 'bleu', 'bleu4', 'rouge', 'rougeL', 'meteor', 'chrfpp', 'bertscore_f1']].sort_values('bleu', ascending=False).to_string(index=False))
# print("="*115)